# EXPERIMENTOS DEL MODELADO

Este cuaderno resume los resultados generados por `src/models/run_experiments.py`. Partimos de los archivos guardados (métricas, resumen, predicciones y modelos) para:

1. Visualizar y comparar las métricas por modelo/fold.
2. Analizar los errores en el conjunto de test (residuos y dispersión y_true vs. y_pred).
3. Revisar la importancia de características (cuando el modelo lo permite) y documentar conclusiones.

### 1. PREPARACIÓN.
Actualiza la ruta `EXPERIMENT_BASE_DIR` con la carpeta específica (p. ej., `experiment_YYYYMMDD_HHMMSS`).


In [17]:
import json
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import joblib
import numpy as np

EXPERIMENT_BASE_DIR = Path("../data/results/modeling/experiments")

runner_experiments = sorted(
    [d for d in EXPERIMENT_BASE_DIR.iterdir() if d.is_dir() and d.name.startswith("runner_id_20260103")],
    reverse=True,
)
if not runner_experiments:
    raise FileNotFoundError("Runner ID experiment directories not found.")

selected_dir = runner_experiments[0]
print(f"Using experiment directory: {selected_dir}")

metrics_path = selected_dir / "metrics.csv"
summary_path = selected_dir / "summary.csv"
predictions_path = selected_dir / "predictions.parquet"
feature_cols_path = selected_dir / "feature_columns.json"

if not metrics_path.exists() or not summary_path.exists():
    raise FileNotFoundError("Required metrics.csv or summary.csv not found in the selected experiment directory.")

experiment_data = {
    "dir": selected_dir,
    "metrics": pd.read_csv(metrics_path),
    "summary": pd.read_csv(summary_path),
}

if predictions_path.exists():
    experiment_data["pred"] = pd.read_parquet(predictions_path)
else:
    print("predictions.parquet not found; setting predictions to None.")
    experiment_data["pred"] = None

if feature_cols_path.exists():
    experiment_data["feature_columns"] = json.loads(feature_cols_path.read_text())
else:
    experiment_data["feature_columns"] = None

Using experiment directory: ../data/results/modeling/experiments/runner_id_20260103_180841


In [13]:
metrics_table = (
    experiment_data["summary"]
    .pivot_table(
        index="model",
        columns="split",
        values=["mae_mean", "rmse_mean", "r2_mean"],
    )
    .round(4)
)

display(metrics_table)


mae_mean         r2_mean         rmse_mean        
split                        cv    test      cv    test        cv    test
model                                                                    
catboost                 0.0700  0.0543  0.6595  0.8414    0.0966  0.0732
elasticnet               0.0764  0.0761  0.6359  0.6852    0.1005  0.1032
gradient_boosting        0.0687  0.0557  0.6728  0.8323    0.0945  0.0753
hist_gradient_boosting   0.0690  0.0543  0.6659  0.8379    0.0957  0.0741
random_forest            0.0694  0.0576  0.6687  0.8246    0.0950  0.0770
xgboost                  0.0705  0.0569  0.6595  0.8292    0.0966  0.0760

### 2. COMPARATIVA DE MÉTRICAS.
Gráficas para comparar MAE/RMSE/R² por modelo y partición.


In [14]:
summary_df = experiment_data["summary"].copy()
print(f"Experimento: {experiment_data['dir'].name}")

fig_mae = px.bar(
    summary_df,
    x="model",
    y="mae_mean",
    color="split",
    error_y="mae_std",
    title=f"MAE por modelo y partición",
    labels={"model": "Modelo", "split": "Partición", "mae_mean": "MAE"},
    text_auto=".4f"   
)
fig_mae.update_traces(textposition="outside", cliponaxis=False)
fig_mae.show()

fig_rmse = px.bar(
    summary_df,
    x="model",
    y="rmse_mean",
    color="split",
    error_y="rmse_std",
    title=f"RMSE por modelo y partición",
    labels={"model": "Modelo", "split": "Partición", "rmse_mean": "RMSE"},
    text_auto=".4f"
)
fig_rmse.update_traces(textposition="outside", cliponaxis=False)
fig_rmse.show()

fig_r2 = px.bar(
    summary_df,
    x="model",
    y="r2_mean",
    color="split",
    error_y="r2_std",
    title=f"R² por modelo y partición",
    labels={"model": "Modelo", "split": "Partición", "r2_mean": "R²"},
    text_auto=".4f"
)
fig_r2.update_traces(textposition="outside", cliponaxis=False)
fig_r2.show()


Experimento: runner_id_20260103_180841


### 3. RESIDUOS Y DISPERSIONES.

Inspeccionamos cómo se comporta cada modelo sobre el conjunto de test.


In [15]:
pred_df = experiment_data["pred"].copy()
pred_df["residual"] = pred_df["y_true"] - pred_df["y_pred"]

fig_scatter = px.scatter(
    pred_df,
    x="y_true",
    y="y_pred",
    color="model",
    title=f"Valor real vs. predicho (test)",
    labels={"y_true": f"Valor real", "y_pred": f"Valor predicho", "model": "Modelo"}
)
fig_scatter.add_trace(
    go.Scatter(
        x=[pred_df.y_true.min(), pred_df.y_true.max()],
        y=[pred_df.y_true.min(), pred_df.y_true.max()],
        mode="lines",
        name="Ideal",
        line=dict(dash="dash", color="gray")
    )
)
fig_scatter.show()

fig_res = px.box(
    pred_df,
    x="model",
    y="residual",
    title=f"Distribución de residuos (test)",
    labels={"model": "Modelo", "residual": "Residuo"}
)
fig_res.update_traces(boxmean=True)
fig_res.show()



### 4. IMPORTANCIA DE CARACTERÍSTICAS.

Analizamos la contribución de las 20 características más relevantes para cada modelo que expone importancias o coeficientes, a fin de documentar qué señales fisiológicas o biomecánicas dominan las predicciones.

In [16]:
feature_columns = experiment_data["feature_columns"]
available_models = sorted(experiment_data["summary"]["model"].unique())
print(f"Experimento: {experiment_data['dir'].name})")

for model_name in available_models:
    model_path = experiment_data["dir"] / f"{model_name}_best.joblib"
    if not model_path.exists():
        print(f"Artefacto del modelo no encontrado para {model_name}.")
        continue

    pipeline = joblib.load(model_path)
    model = pipeline.named_steps["model"]
    importances = getattr(model, "feature_importances_", None)

    if importances is None:
        coef = getattr(model, "coef_", None)
        if coef is not None:
            importances = np.abs(np.ravel(coef))
        else:
            print(f"  {model_name} no expone importancias ni coeficientes; se omite.")
            continue

    feature_names = (
        feature_columns
        if feature_columns is not None and len(feature_columns) == len(importances)
        else [f"feature_{i}" for i in range(len(importances))]
    )

    fi = (
        pd.DataFrame({"feature": feature_names, "importance": importances})
        .sort_values("importance", ascending=False)
        .head(20)
    )

    fig = px.bar(
        fi,
        x="feature",
        y="importance",
        title=f"20 características más relevantes ({model_name})",
        labels={"feature": "Característica", "importance": "Importancia"}
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()


Experimento: runner_id_20260103_180841)


  hist_gradient_boosting no expone importancias ni coeficientes; se omite.
